In [35]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MMDS") \
    .master("local[*]") \
    .getOrCreate()

## Users

In [45]:
from pyspark.sql.functions import col

users_df = spark.read.option("delimiter", "::").csv("./data/users.dat")
users_df = users_df.toDF(
    "user_id",
    "gender",
    "age",
    "occupation",
    "zip_code",
)

users = (
    users_df
    .withColumn("user_id", col("user_id").cast("int"))
    .withColumn("age", col("age").cast("int"))
    .withColumn("occupation", col("occupation").cast("int"))
)

users.printSchema()
users.show()

root
 |-- user_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- occupation: integer (nullable = true)
 |-- zip_code: string (nullable = true)

+-------+------+---+----------+--------+
|user_id|gender|age|occupation|zip_code|
+-------+------+---+----------+--------+
|      1|     F|  1|        10|   48067|
|      2|     M| 56|        16|   70072|
|      3|     M| 25|        15|   55117|
|      4|     M| 45|         7|   02460|
|      5|     M| 25|        20|   55455|
|      6|     F| 50|         9|   55117|
|      7|     M| 35|         1|   06810|
|      8|     M| 25|        12|   11413|
|      9|     M| 25|        17|   61614|
|     10|     F| 35|         1|   95370|
|     11|     F| 25|         1|   04093|
|     12|     M| 25|        12|   32793|
|     13|     M| 45|         1|   93304|
|     14|     M| 35|         0|   60126|
|     15|     M| 25|         7|   22903|
|     16|     F| 35|         0|   20670|
|     17|     M| 5

In [43]:
print(f"There are {users.count()} users")

There are 6040 users


In [88]:
from pyspark.sql.functions import sum, when

users.select([
    sum(when(col(c).isNull(), 1)).alias(c) for c in users.columns
]).show()

+-------+------+----+----------+--------+
|user_id|gender| age|occupation|zip_code|
+-------+------+----+----------+--------+
|   NULL|  NULL|NULL|      NULL|    NULL|
+-------+------+----+----------+--------+



In [ ]:
users.groupBy("gender").count().orderBy("count", ascending=False).show()

+------+-----+
|gender|count|
+------+-----+
|     M| 4331|
|     F| 1709|
+------+-----+



In [90]:
users.groupBy("age").count().orderBy("count", ascending=False).show()

+---+-----+
|age|count|
+---+-----+
| 25| 2096|
| 35| 1193|
| 18| 1103|
| 45|  550|
| 50|  496|
| 56|  380|
|  1|  222|
+---+-----+



### Occupation

In [93]:
occupation_data = [
    (0,  "other"),
    (1,  "academic/educator"),
    (2,  "artist"),
    (3,  "clerical/admin"),
    (4,  "college/grad student"),
    (5,  "customer service"),
    (6,  "doctor/health care"),
    (7,  "executive/managerial"),
    (8,  "farmer"),
    (9,  "homemaker"),
    (10, "K-12 student"),
    (11, "lawyer"),
    (12, "programmer"),
    (13, "retired"),
    (14, "sales/marketing"),
    (15, "scientist"),
    (16, "self-employed"),
    (17, "technician/engineer"),
    (18, "tradesman/craftsman"),
    (19, "unemployed"),
    (20, "writer"),
]

occupation_df = spark.createDataFrame(
    occupation_data,
    ["occupation", "occupation_name"]
)

occupation_df.show()

+----------+--------------------+
|occupation|     occupation_name|
+----------+--------------------+
|         0|               other|
|         1|   academic/educator|
|         2|              artist|
|         3|      clerical/admin|
|         4|college/grad student|
|         5|    customer service|
|         6|  doctor/health care|
|         7|executive/managerial|
|         8|              farmer|
|         9|           homemaker|
|        10|        K-12 student|
|        11|              lawyer|
|        12|          programmer|
|        13|             retired|
|        14|     sales/marketing|
|        15|           scientist|
|        16|       self-employed|
|        17| technician/engineer|
|        18| tradesman/craftsman|
|        19|          unemployed|
+----------+--------------------+
only showing top 20 rows


In [96]:
users_enriched = users.join(occupation_df, on='occupation', how='left')
users_enriched.show()

+----------+-------+------+---+--------+--------------------+
|occupation|user_id|gender|age|zip_code|     occupation_name|
+----------+-------+------+---+--------+--------------------+
|         0|     14|     M| 35|   60126|               other|
|         0|     16|     F| 35|   20670|               other|
|         7|      4|     M| 45|   02460|executive/managerial|
|         7|     15|     M| 25|   22903|executive/managerial|
|         9|      6|     F| 50|   55117|           homemaker|
|        17|      9|     M| 25|   61614| technician/engineer|
|         1|      7|     M| 35|   06810|   academic/educator|
|         1|     10|     F| 35|   95370|   academic/educator|
|         1|     11|     F| 25|   04093|   academic/educator|
|         1|     13|     M| 45|   93304|   academic/educator|
|         1|     17|     M| 50|   95350|   academic/educator|
|        10|      1|     F|  1|   48067|        K-12 student|
|        10|     19|     M|  1|   48073|        K-12 student|
|       

In [97]:
users_enriched.groupBy("occupation_name").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|     occupation_name|count|
+--------------------+-----+
|college/grad student|  759|
|               other|  711|
|executive/managerial|  679|
|   academic/educator|  528|
| technician/engineer|  502|
|          programmer|  388|
|     sales/marketing|  302|
|              writer|  281|
|              artist|  267|
|       self-employed|  241|
|  doctor/health care|  236|
|        K-12 student|  195|
|      clerical/admin|  173|
|           scientist|  144|
|             retired|  142|
|              lawyer|  129|
|    customer service|  112|
|           homemaker|   92|
|          unemployed|   72|
| tradesman/craftsman|   70|
+--------------------+-----+
only showing top 20 rows


In [100]:
users.groupBy("gender", "age").count().orderBy("count", ascending=False).show()

+------+---+-----+
|gender|age|count|
+------+---+-----+
|     M| 25| 1538|
|     M| 35|  855|
|     M| 18|  805|
|     F| 25|  558|
|     M| 45|  361|
|     M| 50|  350|
|     F| 35|  338|
|     F| 18|  298|
|     M| 56|  278|
|     F| 45|  189|
|     F| 50|  146|
|     M|  1|  144|
|     F| 56|  102|
|     F|  1|   78|
+------+---+-----+



In [104]:
users_enriched.groupBy("gender", "occupation_name").count().orderBy("count", ascending=False).show()

+------+--------------------+-----+
|gender|     occupation_name|count|
+------+--------------------+-----+
|     M|executive/managerial|  540|
|     M|college/grad student|  525|
|     M|               other|  479|
|     M| technician/engineer|  450|
|     M|          programmer|  338|
|     M|   academic/educator|  319|
|     F|college/grad student|  234|
|     F|               other|  232|
|     M|     sales/marketing|  223|
|     F|   academic/educator|  209|
|     M|              writer|  203|
|     M|       self-employed|  190|
|     M|              artist|  176|
|     F|executive/managerial|  139|
|     M|  doctor/health care|  134|
|     M|        K-12 student|  129|
|     M|           scientist|  116|
|     M|             retired|  108|
|     M|              lawyer|  107|
|     F|  doctor/health care|  102|
+------+--------------------+-----+
only showing top 20 rows


In [110]:
users.groupBy("user_id").count().filter("count > 1").show()

+-------+-----+
|user_id|count|
+-------+-----+
+-------+-----+



## Movies

In [142]:
from pyspark.sql.functions import split, substring

movies_df = spark.read.option("delimiter", "::").csv("./data/movies.dat")
movies_df = movies_df.toDF(
    "movie_id",
    "title",
    "genres",
)

movies = (
    movies_df
    .withColumn("movie_id", col("movie_id").cast("int"))
    .withColumn("genres", split("genres", r"\|"))
    .withColumn("year", substring("title", -5, 4).cast("int"))
)

movies.show(truncate=False)
movies.printSchema()

+--------+-------------------------------------+--------------------------------+----+
|movie_id|title                                |genres                          |year|
+--------+-------------------------------------+--------------------------------+----+
|1       |Toy Story (1995)                     |[Animation, Children's, Comedy] |1995|
|2       |Jumanji (1995)                       |[Adventure, Children's, Fantasy]|1995|
|3       |Grumpier Old Men (1995)              |[Comedy, Romance]               |1995|
|4       |Waiting to Exhale (1995)             |[Comedy, Drama]                 |1995|
|5       |Father of the Bride Part II (1995)   |[Comedy]                        |1995|
|6       |Heat (1995)                          |[Action, Crime, Thriller]       |1995|
|7       |Sabrina (1995)                       |[Comedy, Romance]               |1995|
|8       |Tom and Huck (1995)                  |[Adventure, Children's]         |1995|
|9       |Sudden Death (1995)              

In [113]:
print(f"There are {movies.count()} movies")

There are 3883 movies


In [114]:
from pyspark.sql.functions import sum, when

movies.select([
    sum(when(col(c).isNull(), 1)).alias(c) for c in movies.columns
]).show()

+--------+-----+------+
|movie_id|title|genres|
+--------+-----+------+
|    NULL| NULL|  NULL|
+--------+-----+------+



In [115]:
movies.groupBy("movie_id").count().filter("count > 1").show()

+--------+-----+
|movie_id|count|
+--------+-----+
+--------+-----+



In [143]:
movies.groupBy("year").count().summary().show()

+-------+------------------+-----------------+
|summary|              year|            count|
+-------+------------------+-----------------+
|  count|                81|               81|
|   mean|1959.9382716049383|47.93827160493827|
| stddev|23.628555647252515|81.78635975500627|
|    min|              1919|                1|
|    25%|              1940|               11|
|    50%|              1960|               19|
|    75%|              1980|               35|
|    max|              2000|              345|
+-------+------------------+-----------------+



In [144]:
movies.filter("year is NULL").count()

0

TBD